# Finished Product — Data Cleaning & Normalization

Analyze and normalize the `finished-product-prod.xlsx` reference data from SIC JAC.
Focus on columns `codigo`, `descripcion`, and `SUB-CATEGORIA`.

**Source:** `docs/reference/warehouse/cleaned/finished-product-prod.xlsx`

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

SRC = Path('../../reference/warehouse/cleaned/finished-product-prod.xlsx')

---
## 1. Load & Overview

In [2]:
df = pd.read_excel(SRC)
print(f'Shape: {df.shape}')
df.head(10)

Shape: (818, 13)


,codigo,descripcion,unidad,Saldoin-Cantidad,Saldoin-Valor,ent-Cantidad,ent-Valor,sal-Cantidad,sal-Valor,sadout-Cantidad,saldout-Valor,CATEGORIA,SUB-CATEGORIA
0,01-24-26,ROJO 3017 2/18,KILOS,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
1,01-24-28,CONFITE 2002 2/18,KILOS,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
2,01-24-35,KANTUTA 3045 2/18,KILOS,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
3,01-24-39,CICLAN 2004 2/18,KILOS,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
4,01-24-45-1,ROSADO BB 2001 2/18,KILOS,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
5,01-24-54-1,AMARILLO PATO 0027 2/18,KILOS,11.0,11.0,0.0,0.0,11.0,11.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
6,01-24-58,CARTON AMARILLO 6004 2/18,KILOS,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
7,01-25-01,VICUÑA 6011 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
8,01-25-04,FUCSIA 3049 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18
9,01-25-05,AZUL PETROLEO 7009 2/18,KILOS,0.0,0.0,204.0,204.0,204.0,204.0,0.0,0.0,1 - ALMACEN PRODUCTO TERMINADO,1 - TITULO 2/18


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 818 entries, 0 to 817
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   codigo            818 non-null    str    
 1   descripcion       818 non-null    str    
 2   unidad            818 non-null    str    
 3   Saldoin-Cantidad  818 non-null    float64
 4   Saldoin-Valor     818 non-null    float64
 5   ent-Cantidad      818 non-null    float64
 6   ent-Valor         818 non-null    float64
 7   sal-Cantidad      818 non-null    float64
 8   sal-Valor         818 non-null    float64
 9   sadout-Cantidad   818 non-null    float64
 10  saldout-Valor     818 non-null    float64
 11  CATEGORIA         818 non-null    str    
 12  SUB-CATEGORIA     818 non-null    str    
dtypes: float64(8), str(5)
memory usage: 83.2 KB


In [4]:
df.isnull().sum()

codigo              0
descripcion         0
unidad              0
Saldoin-Cantidad    0
Saldoin-Valor       0
ent-Cantidad        0
ent-Valor           0
sal-Cantidad        0
sal-Valor           0
sadout-Cantidad     0
saldout-Valor       0
CATEGORIA           0
SUB-CATEGORIA       0
dtype: int64

---
## 2. Column Overview

Schema:
- `codigo`: Product code (format XX-YY-ZZ or XX-YY-ZZ-N)
- `descripcion`: Product description with color name, code, and thickness
- `unidad`: Unit of measure (typically KILOS)
- `Saldoin-Cantidad` / `Saldoin-Valor`: Opening balance quantity/value
- `ent-Cantidad` / `ent-Valor`: Entry (entrada) quantity/value
- `sal-Cantidad` / `sal-Valor`: Exit (salida) quantity/value
- `sadout-Cantidad` / `saldout-Valor`: Closing balance quantity/value
- `CATEGORIA`: Product category
- `SUB-CATEGORIA`: Sub-category (usually thickness like '1 - TITULO 2/18')

In [5]:
print('Unique categories:', df['CATEGORIA'].nunique())
print('\nCategory values:')
print(df['CATEGORIA'].value_counts())

Unique categories: 1

Category values:
CATEGORIA
1 - ALMACEN PRODUCTO TERMINADO    818
Name: count, dtype: int64


In [6]:
print('Unique sub-categories:', df['SUB-CATEGORIA'].nunique())
print('\nSub-category values:')
print(df['SUB-CATEGORIA'].value_counts())

Unique sub-categories: 8

Sub-category values:
SUB-CATEGORIA
1 - TITULO 2/18     539
2 - TITULO 2/32     195
33 - TITULO 2/12     39
27 - TITULO  4/9     22
38 - TITULO 2/10     15
45 - STOLL            6
4 - RECUPERADO        1
8 - TIPO CH           1
Name: count, dtype: int64


---
## 3. Column — codigo

Unique identifier per row. Pattern: `XX-YY-ZZ` or `XX-YY-ZZ-N`

In [7]:
print(f'Unique codes: {df["codigo"].nunique()} out of {len(df)} rows')
print(f'Any nulls: {df["codigo"].isnull().any()}')

codes = df['codigo'].astype(str)
print('\nSample codes:')
print(codes.head(15).to_list())
print(f'\nCodes longer than 8 chars: {(codes.str.len() > 8).sum()}')
if (codes.str.len() > 8).any():
    print(codes[codes.str.len() > 8].unique())

Unique codes: 818 out of 818 rows
Any nulls: False

Sample codes:
['01-24-26', '01-24-28', '01-24-35', '01-24-39', '01-24-45-1', '01-24-54-1', '01-24-58', '01-25-01', '01-25-04', '01-25-05', '01-25-07', '01-25-09', '01-25-11', '01-25-12', '01-25-13']

Codes longer than 8 chars: 31
<StringArray>
[ '01-24-45-1',  '01-24-54-1',  '02-25-41-1',  '03-24-03-1',  '03-24-46-1',
  '12-24-47-1',  '15-24-08-1',  '15-24-57-1',  '16-24-48-1',  '17-24-69-1',
  '17-24-71-1',  '18-24-48-1', '27-45-24-16', '27-45-24-39', '27-45-24-40',
 '27-45-24-41', '27-45-24-42', '27-45-24-44', '27-45-24-45',  '38-23-54-1',
  '39-23-22-1',  '13-24-27-1', '27-45-24-01', '27-45-24-02', '27-45-24-07',
 '27-45-24-37', '27-45-24-43',  '40-23-85-1', '37-27-24-04', '02-20-24-08',
 '13-20-24-01']
Length: 31, dtype: str


---
## 4. Column — descripcion (main focus)

String to normalize. Expected pattern:
```
COLOR_NAME  COLOR_CODE  THICKNESS
```
Separated by double spaces. Examples:
- `ROJO  3017  2/18` → name=ROJO, code=3017, thickness=2/18
- `AZUL PETROLEO  7009  2/18` → name=AZUL PETROLEO, code=7009, thickness=2/18
- `CARTON AMARILLO  6004  2/18`

In [8]:
descs = df['descripcion'].astype(str)
print(f'Unique descriptions: {descs.nunique()} out of {len(descs)}')
print(f'Nulls: {descs.isnull().sum()}')

# Look for double-space separator
has_double_space = descs.str.contains(r'  ')
print(f'\nHas double-space separator: {has_double_space.sum()} / {len(descs)}')
print(f'Without double-space: {descs[~has_double_space].unique()}')

Unique descriptions: 249 out of 818
Nulls: 0

Has double-space separator: 818 / 818
Without double-space: <StringArray>
[]
Length: 0, dtype: str


In [9]:
# Density: how many values per description?
parts = descs.str.split(r'\s{2,}')
print('Breakdown by number of parts (split by double-space):')
print(parts.str.len().value_counts().sort_index())

print('\n=== 2-part descriptions ===')
print(descs[parts.str.len() == 2].unique())

print('\n=== 3-part descriptions (first 10) ===')
three_part = descs[parts.str.len() == 3].unique()
for d in sorted(three_part)[:10]:
    print(' ', d)

Breakdown by number of parts (split by double-space):
descripcion
2      2
3    813
4      3
Name: count, dtype: int64

=== 2-part descriptions ===
<StringArray>
['VICUÑA  6012 2/32-D', 'FUCSIA 3009  2/32']
Length: 2, dtype: str

=== 3-part descriptions (first 10) ===
  AM. CANARIO  0015  4/9
  AM. CANARIO  0021  2/18
  AMARILLO  0023  2/32-D
  AMARILLO CANARIO  0021  2/18
  AMARILLO ORO  0016  2/18
  AMARILLO ORO  0016  2/32
  AMARILLO ORO  0016  2/32-D
  AMARILLO PATO  0027  2/18
  API  7032  2/18
  ARENA  6001  2/12


In [10]:
# Explore unique patterns
import random
unique_descs = sorted(descs.unique())
print(f'Total unique descriptions: {len(unique_descs)}')
print('\nRandom sample:')
for d in random.sample(unique_descs, min(20, len(unique_descs))):
    print(f'  {d}')

Total unique descriptions: 249

Random sample:
  AVIACION  7042  2/18
  BLANCO  0010  2/10
  GRIS  400  2/12
  VERDE LIMON  4051  2/32
  AZALEA  7039  4/9
  ROJO  3006  2/18
  PERLA  0031  2/12
  BLANCO  0010  2/32-D
  VICUÑA  6068  2/18
  VICUÑA  6007  2/10
  LILA BAJO  7018  2/32
  LIMON  0020  2/18
  AMARILLO  0023  2/32-D
  AZUL PASTEL  7012  2/18
  AMARILLO ORO  0016  2/32-D
  PLOMO OSC.  271  2/12
  AZUL PETROLEO  7084  2/18
  BIEGE OC.  6021  2/18
  NARANJA  1023  2/18
  CLAVEL  3034  2/18


---
## 5. Parse `descripcion`

Separate the description into `product_name`, `color_code`, `thickness`.

In [11]:
pattern = re.compile(r'^(?P<product>.+?)\s+(?P<color_code>\d{3,4})\s+(?P<thickness>\d+/\d+)$', re.IGNORECASE)

matched = descs.str.contains(pattern, na=False)
print('Matched rows:', matched.sum())
print('Total rows:', len(descs))
print('\nMatched rate:', f'{matched.sum() / len(descs) * 100:.1f}%')

Matched rows: 656
Total rows: 818

Matched rate: 80.2%


C:\Users\Alexander\AppData\Local\Temp\ipykernel_15308\445436964.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matched = descs.str.contains(pattern, na=False)


In [12]:
unmatched = df[~descs.str.contains(pattern, na=False)]
print('Unmatched sample count:', len(unmatched))
if len(unmatched) > 0:
    unmatched[['descripcion']].head(20)

Unmatched sample count: 162


C:\Users\Alexander\AppData\Local\Temp\ipykernel_15308\3563395348.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  unmatched = df[~descs.str.contains(pattern, na=False)]


In [13]:
def parse_product_desc(description):
    if not isinstance(description, str) or not description.strip():
        return pd.Series({
            'product_name': None,
            'color_code': None,
            'thickness': None,
        })

    text = description.strip()
    match = pattern.match(text)
    if match:
        return pd.Series({
            'product_name': match.group('product').strip(),
            'color_code': match.group('color_code').strip(),
            'thickness': match.group('thickness').strip(),
        })

    # Fallback: split by multiple spaces
    parts = re.split(r'\s{2,}', text)
    product_name = None
    color_code = None
    thickness = None
    
    if len(parts) >= 3 and re.match(r'^\d{3,4}$', parts[-2]) and re.match(r'^\d+/\d+$', parts[-1]):
        product_name = ' '.join(parts[:-2])
        color_code = parts[-2]
        thickness = parts[-1]
    elif len(parts) >= 2 and re.match(r'^\d+/\d+$', parts[-1]):
        product_name = ' '.join(parts[:-1])
        thickness = parts[-1]
    else:
        product_name = text

    return pd.Series({
        'product_name': product_name,
        'color_code': color_code,
        'thickness': thickness,
    })

parsed_desc = df['descripcion'].apply(parse_product_desc)
df = pd.concat([df, parsed_desc], axis=1)
df[['descripcion', 'product_name', 'color_code', 'thickness']].head(15)

,descripcion,product_name,color_code,thickness
0,ROJO 3017 2/18,ROJO,3017,2/18
1,CONFITE 2002 2/18,CONFITE,2002,2/18
2,KANTUTA 3045 2/18,KANTUTA,3045,2/18
3,CICLAN 2004 2/18,CICLAN,2004,2/18
4,ROSADO BB 2001 2/18,ROSADO BB,2001,2/18
5,AMARILLO PATO 0027 2/18,AMARILLO PATO,0027,2/18
6,CARTON AMARILLO 6004 2/18,CARTON AMARILLO,6004,2/18
7,VICUÑA 6011 2/18,VICUÑA,6011,2/18
8,FUCSIA 3049 2/18,FUCSIA,3049,2/18
9,AZUL PETROLEO 7009 2/18,AZUL PETROLEO,7009,2/18


In [14]:
print('Parsed totals:')
print('Missing product_name:', df['product_name'].isnull().sum())
print('Missing color_code:', df['color_code'].isnull().sum())
print('Missing thickness:', df['thickness'].isnull().sum())

Parsed totals:
Missing product_name: 0
Missing color_code: 162
Missing thickness: 161


In [15]:
unparsed = df[df['color_code'].isnull() | df['thickness'].isnull()]
print(f'Unparsed rows: {len(unparsed)}')
if len(unparsed) > 0:
    unparsed[['descripcion', 'product_name', 'color_code', 'thickness']].head(20)

Unparsed rows: 162


---
## 6. Validate quantity columns

Confirm that opening balance, entry, exit, and closing balance behave as expected.

In [16]:
qty_cols = ['Saldoin-Cantidad', 'ent-Cantidad', 'sal-Cantidad', 'sadout-Cantidad']
for col in qty_cols:
    if col in df.columns:
        print(f'{col}:')
        print(f'  min: {df[col].min()}, max: {df[col].max()}')
        print(f'  nulls: {df[col].isnull().sum()}')
    else:
        print(f'{col}: missing')

Saldoin-Cantidad:
  min: 0.0, max: 204.0
  nulls: 0
ent-Cantidad:
  min: 0.0, max: 205.0
  nulls: 0
sal-Cantidad:
  min: 0.0, max: 205.0
  nulls: 0
sadout-Cantidad:
  min: 0.0, max: 204.0
  nulls: 0


In [17]:
# Check balance logic: opening + entry - exit = closing
if all(col in df.columns for col in ['Saldoin-Cantidad', 'ent-Cantidad', 'sal-Cantidad', 'sadout-Cantidad']):
    df['_balance_check'] = df['Saldoin-Cantidad'] + df['ent-Cantidad'] - df['sal-Cantidad']
    consistent = (df['_balance_check'] == df['sadout-Cantidad'])
    print(f'Balance consistent: {consistent.sum()} / {len(df)}')
    
    inconsistent = df[~consistent]
    if len(inconsistent) > 0:
        print(f'\nInconsistent rows (sample):')
        print(inconsistent[['codigo', 'Saldoin-Cantidad', 'ent-Cantidad', 'sal-Cantidad', 'sadout-Cantidad', '_balance_check']].head(10))

Balance consistent: 818 / 818


---
## 7. Proposal: Normalized schema

| Column | Source | Notes |
|--------|--------|-------|
| `codigo` | `codigo` | Product code |
| `product_name` | parsed `descripcion` | Product/color name |
| `color_code` | parsed `descripcion` | Numeric color code |
| `thickness` | parsed `descripcion` | Yarn thickness/gauge |
| `unidad` | `unidad` | Unit of measure |
| `saldoin_cantidad` | `Saldoin-Cantidad` | Opening balance quantity |
| `saldoin_valor` | `Saldoin-Valor` | Opening balance value |
| `entrada_cantidad` | `ent-Cantidad` | Entry quantity |
| `entrada_valor` | `ent-Valor` | Entry value |
| `salida_cantidad` | `sal-Cantidad` | Exit quantity |
| `salida_valor` | `sal-Valor` | Exit value |
| `saldout_cantidad` | `sadout-Cantidad` | Closing balance quantity |
| `saldout_valor` | `saldout-Valor` | Closing balance value |
| `categoria` | `CATEGORIA` | Product category |
| `sub_categoria` | `SUB-CATEGORIA` | Sub-category |
| `balance_consistent` | calculated | Whether opening + entry - exit = closing |

In [18]:
cleaned = df.copy()
cleaned = cleaned.rename(columns={
    'Saldoin-Cantidad': 'saldoin_cantidad',
    'Saldoin-Valor': 'saldoin_valor',
    'ent-Cantidad': 'entrada_cantidad',
    'ent-Valor': 'entrada_valor',
    'sal-Cantidad': 'salida_cantidad',
    'sal-Valor': 'salida_valor',
    'sadout-Cantidad': 'saldout_cantidad',
    'saldout-Valor': 'saldout_valor',
    'CATEGORIA': 'categoria',
    'SUB-CATEGORIA': 'sub_categoria',
})

# Ensure balance check exists
if '_balance_check' not in cleaned.columns:
    cleaned['_balance_check'] = cleaned['saldoin_cantidad'] + cleaned['entrada_cantidad'] - cleaned['salida_cantidad']

cleaned['balance_consistent'] = cleaned['_balance_check'] == cleaned['saldout_cantidad']

print('Cleaned shape:', cleaned.shape)
print('\nSample:')
cleaned[['codigo', 'product_name', 'color_code', 'thickness', 'saldoin_cantidad', 'entrada_cantidad', 'salida_cantidad', 'saldout_cantidad', 'balance_consistent']].head(10)

Cleaned shape: (818, 18)

Sample:


,codigo,product_name,color_code,thickness,saldoin_cantidad,entrada_cantidad,salida_cantidad,saldout_cantidad,balance_consistent
0,01-24-26,ROJO,3017,2/18,0.5,0.0,0.5,0.0,True
1,01-24-28,CONFITE,2002,2/18,0.5,0.0,0.5,0.0,True
2,01-24-35,KANTUTA,3045,2/18,1.0,0.0,1.0,0.0,True
3,01-24-39,CICLAN,2004,2/18,1.0,0.0,1.0,0.0,True
4,01-24-45-1,ROSADO BB,2001,2/18,0.5,0.0,0.5,0.0,True
5,01-24-54-1,AMARILLO PATO,0027,2/18,11.0,0.0,11.0,0.0,True
6,01-24-58,CARTON AMARILLO,6004,2/18,1.0,0.0,1.0,0.0,True
7,01-25-01,VICUÑA,6011,2/18,0.0,204.0,204.0,0.0,True
8,01-25-04,FUCSIA,3049,2/18,0.0,204.0,204.0,0.0,True
9,01-25-05,AZUL PETROLEO,7009,2/18,0.0,204.0,204.0,0.0,True


In [19]:
print('Balance summary:')
print(cleaned['balance_consistent'].value_counts())
print(f'\nConsistency rate: {cleaned["balance_consistent"].sum() / len(cleaned) * 100:.1f}%')

Balance summary:
balance_consistent
True    818
Name: count, dtype: int64

Consistency rate: 100.0%


---
## 8. Next Steps

- Refine parsing for edge cases (compound names, special characters)
- Investigate balance inconsistencies
- Compare thickness values against sub-categories
- Validate color codes across warehouse datasets
- Export cleaned output to CSV or reference dataset

In [20]:
cleaned_csv_path = Path('finished-product-cleaned.csv')
output_cols = ['codigo', 'product_name', 'color_code', 'thickness', 'unidad', 'saldoin_cantidad', 'saldoin_valor', 'entrada_cantidad', 'entrada_valor', 'salida_cantidad', 'salida_valor', 'saldout_cantidad', 'saldout_valor', 'categoria', 'sub_categoria', 'balance_consistent']
cleaned[output_cols].to_csv(cleaned_csv_path, index=False)
print(f'Exported {len(cleaned)} rows to {cleaned_csv_path}')

Exported 818 rows to finished-product-cleaned.csv
